# 🎲 OBLEOPOLIS — Análisis del Rivalry Ratio
### Metodología basada en la Teoría de Cooperación y Competencia de Morton Deutsch

---
> **Instrucciones:** Coloca `Obleopolis_Registro_completed.xlsx` en la misma carpeta y ejecuta celda por celda con **Shift + Enter**.

### 📐 Fórmula del Rivalry Ratio

$$\text{Ratio}_{YA} = \frac{D_Y}{D_A}$$

| Resultado | Interpretación |
|-----------|----------------|
| **Ratio < 1** | El Objetivo va **más adelantado** → incentivo de ataque |
| **Ratio = 1** | Empate técnico → zona propicia para Trueques |
| **Ratio > 1** | El Activo va **más adelantado** que el Objetivo |

> **Ejemplo:** Activo D=3, Objetivo D=1 → Ratio = 1/3 = **0.33 < 1** → el Objetivo gana → ataque esperado ✅

---

## 0 · Importaciones y Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
    'xtick.labelsize'  : 10,
    'ytick.labelsize'  : 10,
    'figure.facecolor' : '#F8FBFF',
    'axes.facecolor'   : '#FFFFFF',
})

# ── Configuración del experimento ─────────────────────────────────────────
# Solo 3 jugadores participaron en las pruebas
JUGADORES = ['Jugador1', 'Jugador2', 'Jugador3']

PLAYER_COLORS = {
    'Jugador1': '#2E86AB',
    'Jugador2': '#E84855',
    'Jugador3': '#3BB273',
}

print('✅  Librerías cargadas correctamente')
print(f'   Jugadores configurados: {JUGADORES}')

## 1 · Carga y Limpieza del Dataset

In [ ]:
# ── Cambia aquí el nombre del archivo si es necesario ─────────────────────
ARCHIVO = 'Obleopolis_Registro_completed.xlsx'
HOJA    = 'REGISTRO_PARTIDAS'

df_raw = pd.read_excel(ARCHIVO, sheet_name=HOJA, header=1, skiprows=[2])

df_raw.columns = [
    'Ronda_ID', 'Turno_Num', 'Jugador_Activo',
    'D_Jugador1', 'D_Jugador2', 'D_Jugador3', 'D_Jugador4',
    'Accion_Tomada', 'Jugador_Objetivo', 'Carta_Utilizada',
    'Rivalry_Ratio_Excel'
]

# Eliminar filas vacías
df = df_raw.dropna(subset=['Ronda_ID', 'Turno_Num', 'Jugador_Activo']).copy()

# ── CORRECCIÓN: acepta rondas 1-99, no solo 1-6 ───────────────────────────
df['Ronda_ID'] = pd.to_numeric(df['Ronda_ID'], errors='coerce')
df = df[df['Ronda_ID'].notna() & (df['Ronda_ID'] >= 1)].copy()
df['Ronda_ID'] = df['Ronda_ID'].astype(int)

df['Turno_Num'] = pd.to_numeric(df['Turno_Num'], errors='coerce').astype(int)

# ── CORRECCIÓN: solo columnas de 3 jugadores (D_Jugador4 vacía) ───────────
for col in ['D_Jugador1', 'D_Jugador2', 'D_Jugador3']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(4).astype(int)

# D_Jugador4 no fue usado — se descarta sin afectar el análisis
df.drop(columns=['D_Jugador4'], inplace=True, errors='ignore')

df['Jugador_Activo']   = df['Jugador_Activo'].astype(str).str.strip()
df['Accion_Tomada']    = df['Accion_Tomada'].astype(str).str.strip()
df['Jugador_Objetivo'] = df['Jugador_Objetivo'].astype(str).str.strip().replace('nan', 'N/A')
df['Carta_Utilizada']  = df['Carta_Utilizada'].astype(str).str.strip()

# Solo filas con jugadores válidos
df = df[df['Jugador_Activo'].isin(JUGADORES)].copy()
df.reset_index(drop=True, inplace=True)

print(f'✅  Dataset cargado correctamente')
print(f'   Turnos totales : {len(df)}')
print(f'   Rondas         : {df["Ronda_ID"].nunique()} ({df["Ronda_ID"].min()}–{df["Ronda_ID"].max()})')
print(f'   Jugadores      : {sorted(df["Jugador_Activo"].unique())}')
print()
df.head(10)

## 2 · Cálculo del Rivalry Ratio

$$\text{Ratio}_{YA} = \frac{D_Y}{D_A}$$

In [ ]:
# ── CORRECCIÓN: DIST_MAP solo con 3 jugadores ─────────────────────────────
DIST_MAP = {
    'Jugador1': 'D_Jugador1',
    'Jugador2': 'D_Jugador2',
    'Jugador3': 'D_Jugador3',
}

def get_distancia(row, jugador_col):
    nombre = row[jugador_col]
    if nombre not in DIST_MAP:
        return np.nan
    return row[DIST_MAP[nombre]]

def calcular_ratio(row):
    """Rivalry Ratio = DY / DA.
    Ratio < 1 → Objetivo va ganando → el Activo tiene incentivo de atacarlo.
    """
    if row['Jugador_Objetivo'] in ('N/A', 'nan', '', 'NaN', 'None'):
        return np.nan
    if row['Jugador_Activo'] not in DIST_MAP or row['Jugador_Objetivo'] not in DIST_MAP:
        return np.nan
    DA = get_distancia(row, 'Jugador_Activo')
    DY = get_distancia(row, 'Jugador_Objetivo')
    if pd.isna(DA) or pd.isna(DY):
        return np.nan
    if DA == 0:           # El Activo ya ganó — caso extremo
        return np.inf
    return round(DY / DA, 4)

df['DA'] = df.apply(lambda r: get_distancia(r, 'Jugador_Activo'),   axis=1)
df['DY'] = df.apply(lambda r: get_distancia(r, 'Jugador_Objetivo'), axis=1)
df['Rivalry_Ratio'] = df.apply(calcular_ratio, axis=1)

def interpretar_ratio(r):
    if pd.isna(r) or np.isinf(r): return 'N/A'
    if r < 1:  return 'Objetivo gana (<1)'
    if r > 1:  return 'Activo gana (>1)'
    return 'Empate (=1)'

df['Ratio_Label'] = df['Rivalry_Ratio'].apply(interpretar_ratio)

print('✅  Rivalry Ratio calculado (DY / DA)')
print(f'   Turnos con ratio calculado : {df["Rivalry_Ratio"].notna().sum()}')
print(f'   Turnos sin objetivo (N/A)  : {(df["Jugador_Objetivo"]=="N/A").sum()}')
print()
print('Distribución de estados de rivalidad:')
print(df['Ratio_Label'].value_counts())
print()
df[['Ronda_ID','Turno_Num','Jugador_Activo','DA',
    'Jugador_Objetivo','DY','Rivalry_Ratio','Ratio_Label','Accion_Tomada']].head(12)

## 3 · Estadísticas Descriptivas

In [ ]:
print('='*65)
print('  RESUMEN ESTADÍSTICO — OBLEOPOLIS (Datos reales)')
print('='*65)

print(f'\nTotal de turnos registrados : {len(df)}')
print(f'Rondas completadas          : {df["Ronda_ID"].nunique()} (rondas {df["Ronda_ID"].min()}–{df["Ronda_ID"].max()})')
print(f'Jugadores participantes     : {len(JUGADORES)} (Jugador4 no participó en las pruebas)')

print(f'\nDistribución de acciones:')
acc = df['Accion_Tomada'].value_counts()
for accion, n in acc.items():
    print(f'  {accion:<15} : {n:>4}  ({n/len(df)*100:.1f}%)')

rr = df['Rivalry_Ratio'].replace([np.inf, -np.inf], np.nan).dropna()
print(f'\nRivalry Ratio (DY/DA) — {len(rr)} turnos con objetivo:')
print(f'  Media   : {rr.mean():.3f}')
print(f'  Mediana : {rr.median():.3f}')
print(f'  Mín     : {rr.min():.3f}')
print(f'  Máx     : {rr.max():.3f}')
print(f'  Desv.St : {rr.std():.3f}')
n_bajo = (rr < 1).sum()
print(f'\n  Ratio < 1 (Objetivo iba ganando) : {n_bajo} ({n_bajo/len(rr)*100:.1f}%)')
print(f'  Ratio = 1 (Empate)               : {(rr==1).sum()} ({(rr==1).sum()/len(rr)*100:.1f}%)')
print(f'  Ratio > 1 (Activo iba ganando)   : {(rr>1).sum()} ({(rr>1).sum()/len(rr)*100:.1f}%)')
print('='*65)

---
## 🔴 Análisis A — Correlación entre Rivalry Ratio y Ataques (Competencia)

**Hipótesis:** Los ataques (Obstrucción) se concentran cuando el Rivalry Ratio **< 1** — es decir, cuando el Objetivo va más adelantado que el Activo ($D_Y < D_A$).

In [ ]:
df_plot = df[
    df['Rivalry_Ratio'].notna() &
    ~np.isinf(df['Rivalry_Ratio']) &
    df['Accion_Tomada'].isin(['Obstruccion','Locomocion','Defensa','Cooperacion','Descarte'])
].copy()

accion_palette = {
    'Obstruccion' : '#E84855',
    'Locomocion'  : '#2E86AB',
    'Defensa'     : '#3BB273',
    'Cooperacion' : '#F4A261',
    'Descarte'    : '#AAAAAA',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle(
    'Análisis A — Rivalry Ratio (DY/DA) vs Tipo de Acción\n'
    'Ratio < 1 = Objetivo va ganando → incentivo de ataque (Deutsch) | 3 jugadores, 11 rondas',
    fontsize=12, fontweight='bold', y=1.02
)

# ── A-1: Boxplot por Tipo de Acción ───────────────────────────────────────
ax1 = axes[0]
order = [a for a in ['Obstruccion','Cooperacion','Locomocion','Defensa','Descarte']
         if a in df_plot['Accion_Tomada'].unique()]

if len(df_plot):
    sns.boxplot(
        data=df_plot, x='Accion_Tomada', y='Rivalry_Ratio',
        order=order, palette=accion_palette,
        linewidth=1.4, fliersize=3, ax=ax1
    )
    ymax = min(df_plot['Rivalry_Ratio'].quantile(0.97) + 0.3, 5)
    ax1.axhline(1, color='black', linestyle='--', linewidth=1.3, alpha=0.8, label='Ratio = 1 (empate)')
    ax1.axhspan(0,    1,    alpha=0.07, color='red',   label='< 1: Objetivo gana → ataques esperados')
    ax1.axhspan(1, ymax,    alpha=0.07, color='green', label='> 1: Activo gana')
    ax1.set_ylim(bottom=0, top=ymax)
    ax1.set_title('Distribución del Ratio por Tipo de Acción', fontsize=12)
    ax1.set_xlabel('Tipo de Acción')
    ax1.set_ylabel('Rivalry Ratio  (DY / DA)')
    ax1.legend(fontsize=8)
    ax1.tick_params(axis='x', rotation=15)

# ── A-2: Scatter Ratio vs DA ───────────────────────────────────────────────
ax2 = axes[1]
if len(df_plot):
    for accion, group in df_plot.groupby('Accion_Tomada'):
        ax2.scatter(
            group['DA'] + np.random.uniform(-0.1, 0.1, len(group)),
            group['Rivalry_Ratio'],
            color=accion_palette.get(accion, '#888888'),
            alpha=0.5, edgecolors='none',
            s=40, label=accion, zorder=3
        )
    ax2.axhline(1, color='black', linestyle='--', linewidth=1.3, alpha=0.8, label='Ratio = 1')
    ax2.axhspan(0, 1, alpha=0.07, color='red')
    ax2.set_title('Rivalry Ratio vs Distancia del Jugador Activo\n'
                  '(Ataques en rojo deberían concentrarse bajo la línea)', fontsize=11)
    ax2.set_xlabel('DA — Distancia del Jugador Activo')
    ax2.set_ylabel('Rivalry Ratio  (DY / DA)')
    ax2.set_xticks([0,1,2,3,4])
    ax2.set_ylim(bottom=0)
    ax2.legend(title='Acción', fontsize=8, title_fontsize=9)

plt.tight_layout()
plt.savefig('analisis_A_rivalry_ratio.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊  Gráfico A guardado como analisis_A_rivalry_ratio.png')

In [ ]:
# ── Tabla de apoyo: ataques por rango de Ratio ────────────────────────────
print('📋  Ataques (Obstrucción) — ¿en qué rango de Ratio ocurren?')
atk = df[
    (df['Accion_Tomada'] == 'Obstruccion') &
    df['Rivalry_Ratio'].notna() &
    ~np.isinf(df['Rivalry_Ratio'])
].copy()

if len(atk):
    bajo  = (atk['Rivalry_Ratio'] < 1).sum()
    igual = (atk['Rivalry_Ratio'] == 1).sum()
    sobre = (atk['Rivalry_Ratio'] > 1).sum()
    print(f'\n  Total ataques analizados       : {len(atk)}')
    print(f'  Ratio < 1 (Objetivo ganando)  : {bajo:>4}  ({bajo/len(atk)*100:.1f}%)')
    print(f'  Ratio = 1 (Empate técnico)    : {igual:>4}  ({igual/len(atk)*100:.1f}%)')
    print(f'  Ratio > 1 (Activo iba ganando): {sobre:>4}  ({sobre/len(atk)*100:.1f}%)')
    print(f'\n  → Si Ratio<1 supera el 50%, la Hipótesis A queda CONFIRMADA.')
    print()
    print('Ataques agrupados por DA (distancia del atacante):')
    tabla = atk.groupby('DA').agg(
        Total_Ataques = ('Rivalry_Ratio', 'count'),
        Ratio_Medio   = ('Rivalry_Ratio', 'mean'),
        Ratio_Min     = ('Rivalry_Ratio', 'min'),
        Ratio_Max     = ('Rivalry_Ratio', 'max'),
    ).round(3)
    display(tabla)
else:
    print('  (Sin ataques con ratio calculado)')

---
## 🟠 Análisis B — Alianzas Emergentes (Cooperación Implícita)

**Hipótesis:** Cuando un jugador alcanza D≤2, los demás concentran sus ataques sobre él (coalición implícita anti-líder).

In [ ]:
# ── CORRECCIÓN: lider_del_turno usa solo 3 jugadores ──────────────────────
def lider_del_turno(row):
    dists   = {j: row[f'D_{j}'] for j in JUGADORES if f'D_{j}' in row.index}
    if not dists:
        return 'Empate'
    min_d   = min(dists.values())
    lideres = [j for j, d in dists.items() if d == min_d]
    return lideres[0] if len(lideres) == 1 else 'Empate'

df['Lider']   = df.apply(lider_del_turno, axis=1)
df['D_Lider'] = df.apply(
    lambda r: r[f'D_{r["Lider"]}'] if r['Lider'] != 'Empate' and f'D_{r["Lider"]}' in r.index else np.nan,
    axis=1
)

# Solo ataques con jugadores válidos
df_atk_all = df[
    (df['Accion_Tomada'] == 'Obstruccion') &
    df['Jugador_Activo'].isin(JUGADORES) &
    df['Jugador_Objetivo'].isin(JUGADORES)
].copy()

df_lider = df_atk_all[
    (df_atk_all['Lider'] != 'Empate') &
    (df_atk_all['D_Lider'] <= 2)
].copy()

print(f'Total ataques con objetivo válido : {len(df_atk_all)}')
print(f'Ataques con líder claro a D≤2     : {len(df_lider)}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle(
    'Análisis B — Alianzas Emergentes (Cooperación Implícita)\n'
    '¿Los jugadores concentran ataques sobre el líder? | 3 jugadores, 11 rondas',
    fontsize=12, fontweight='bold', y=1.02
)

# ── B-1: Heatmap todos los ataques ────────────────────────────────────────
ax1 = axes[0]
if len(df_atk_all):
    m = pd.crosstab(df_atk_all['Jugador_Activo'], df_atk_all['Jugador_Objetivo'])
    m = m.reindex(index=JUGADORES, columns=JUGADORES, fill_value=0)
    sns.heatmap(m, annot=True, fmt='d', cmap='YlOrRd',
                linewidths=0.5, linecolor='#cccccc',
                ax=ax1, cbar_kws={'label': 'Nº de ataques'},
                annot_kws={'size': 13, 'weight': 'bold'})
    ax1.set_title(f'Todos los ataques ({len(df_atk_all)} total)', fontsize=11)
    ax1.set_xlabel('Jugador Objetivo (recibe el ataque)')
    ax1.set_ylabel('Jugador Activo (lanza el ataque)')
else:
    ax1.text(0.5, 0.5, 'Sin ataques', ha='center', va='center', transform=ax1.transAxes)

# ── B-2: Heatmap con líder a D≤2 ──────────────────────────────────────────
ax2 = axes[1]
if len(df_lider):
    m2 = pd.crosstab(df_lider['Jugador_Activo'], df_lider['Jugador_Objetivo'])
    m2 = m2.reindex(index=JUGADORES, columns=JUGADORES, fill_value=0)
    sns.heatmap(m2, annot=True, fmt='d', cmap='Reds',
                linewidths=0.5, linecolor='#cccccc',
                ax=ax2, cbar_kws={'label': 'Nº de ataques'},
                annot_kws={'size': 13, 'weight': 'bold'})
    ax2.set_title(f'Ataques con líder a D≤2 ({len(df_lider)} total)\n(¿coalición implícita?)', fontsize=11)
    ax2.set_xlabel('Jugador Objetivo (¿es el líder?)')
    ax2.set_ylabel('Jugador Activo')
else:
    ax2.text(0.5, 0.5, 'Sin ataques con\nlíder a D≤2', ha='center', va='center', transform=ax2.transAxes)

plt.tight_layout()
plt.savefig('analisis_B_alianzas.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊  Gráfico B guardado como analisis_B_alianzas.png')

In [ ]:
# ── % de ataques al líder ─────────────────────────────────────────────────
print('📋  ¿Qué porcentaje de los ataques se dirigen al líder?')
if len(df_atk_all):
    df_b = df[df['Accion_Tomada'] == 'Obstruccion'].copy()
    df_b['Es_al_lider'] = df_b.apply(
        lambda r: r['Jugador_Objetivo'] == r['Lider'] and r['Lider'] != 'Empate', axis=1
    )
    resumen = df_b['Es_al_lider'].value_counts(normalize=True)
    resumen.index = resumen.index.map({True: 'Al líder', False: 'A otros'})
    print((resumen * 100).round(1).to_string())
    pct = df_b['Es_al_lider'].mean() * 100
    verif = '✅ CONFIRMADA' if pct >= 50 else '❌ NO CONFIRMADA'
    print(f'\n→ Hipótesis B: {verif}  ({pct:.1f}% de ataques van al líder)')

    # ── Ataques al líder por ronda ─────────────────────────────────────────
    print()
    print('Distribución por ronda (ataques al líder vs total):')
    por_ronda = df_b.groupby('Ronda_ID').agg(
        Total_Ataques = ('Es_al_lider', 'count'),
        Al_Lider      = ('Es_al_lider', 'sum')
    )
    por_ronda['Pct_al_lider'] = (por_ronda['Al_Lider'] / por_ronda['Total_Ataques'] * 100).round(1)
    display(por_ronda)
else:
    print('  (Sin ataques con objetivo válido)')

---
## 🟢 Análisis C — Cooperación Explícita (Trueques)

**Nota:** En las 11 rondas registradas se detectaron **muy pocos trueques** (2 registros). El análisis estadístico es orientativo; los gráficos se generan pero los resultados deben interpretarse con cautela dado el tamaño reducido de la muestra.

In [ ]:
df_truq = df[df['Accion_Tomada'] == 'Cooperacion'].copy()
df_truq['Delta_D'] = (df_truq['DA'] - df_truq['DY']).abs()
df_truq['Estado_Trueque'] = df_truq['Carta_Utilizada'].apply(
    lambda x: 'Rechazado' if 'rechaz' in str(x).lower() else 'Aceptado'
)

print(f'⚠️  Trueques registrados en las pruebas: {len(df_truq)}')
if len(df_truq):
    display(df_truq[['Ronda_ID','Turno_Num','Jugador_Activo','Jugador_Objetivo',
                      'DA','DY','Delta_D','Rivalry_Ratio','Estado_Trueque','Carta_Utilizada']])
    print()
    print('Nota: Con solo', len(df_truq), 'registro(s), la Hipótesis C no puede ser'
          ' confirmada estadísticamente. Se requieren al menos 10 trueques para un análisis robusto.')
else:
    print('No se registraron cooperaciones con objetivo en las pruebas.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    'Análisis C — Cooperación Explícita (Trueques)\n'
    f'⚠️  Muestra reducida: {len(df_truq)} trueque(s) registrado(s) en 11 rondas',
    fontsize=12, fontweight='bold', y=1.02
)
estado_palette = {'Aceptado': '#3BB273', 'Rechazado': '#E84855'}

for ax in axes:
    ax.set_facecolor('#FFFFFF')

# ── C-1: Conteo ───────────────────────────────────────────────────────────
ax1 = axes[0]
if len(df_truq):
    conteo = df_truq['Estado_Trueque'].value_counts()
    bars = ax1.bar(conteo.index, conteo.values,
                   color=[estado_palette.get(e,'#888') for e in conteo.index],
                   edgecolor='white', linewidth=1.5, width=0.4)
    for bar, val in zip(bars, conteo.values):
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03,
                 str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax1.set_title('Trueques por resultado', fontsize=11)
    ax1.set_ylabel('Cantidad')
    ax1.set_ylim(0, max(conteo.values)*1.5)
    ax1.text(0.5, 0.85, '⚠️ Muestra muy pequeña', ha='center', transform=ax1.transAxes,
             fontsize=9, color='gray', style='italic')
else:
    ax1.text(0.5, 0.5, 'Sin trueques\nregistrados', ha='center', va='center',
             transform=ax1.transAxes, fontsize=11)

# ── C-2: Ratio en cada trueque ────────────────────────────────────────────
ax2 = axes[1]
df_truq_r = df_truq[df_truq['Rivalry_Ratio'].notna() & ~np.isinf(df_truq['Rivalry_Ratio'])]
if len(df_truq_r) >= 1:
    colors = [estado_palette.get(e,'#888') for e in df_truq_r['Estado_Trueque']]
    ax2.bar(range(len(df_truq_r)), df_truq_r['Rivalry_Ratio'].values,
            color=colors, edgecolor='white', linewidth=1.5, width=0.4)
    ax2.axhline(1, color='black', linestyle='--', linewidth=1.2, alpha=0.7, label='Ratio = 1')
    labels = [f'R{int(r["Ronda_ID"])}T{int(r["Turno_Num"])}\n{r["Estado_Trueque"][0]}'
              for _, r in df_truq_r.iterrows()]
    ax2.set_xticks(range(len(df_truq_r)))
    ax2.set_xticklabels(labels, fontsize=9)
    ax2.set_title('Rivalry Ratio por trueque\n(R=Ronda, T=Turno, A=Aceptado)', fontsize=10)
    ax2.set_ylabel('Rivalry Ratio (DY/DA)')
    ax2.set_ylim(bottom=0)
    ax2.legend(fontsize=9)
    patches = [mpatches.Patch(color=v, label=k) for k,v in estado_palette.items()]
    ax2.legend(handles=patches, fontsize=9)
else:
    ax2.text(0.5, 0.5, 'Sin datos de ratio\nen trueques', ha='center', va='center',
             transform=ax2.transAxes, fontsize=11)

# ── C-3: Contexto — acciones por ronda ────────────────────────────────────
# Como hay pocos trueques, usamos este espacio para mostrar el mix de acciones
ax3 = axes[2]
acc_ronda = df.groupby(['Ronda_ID','Accion_Tomada']).size().unstack(fill_value=0)
acc_ronda = acc_ronda.reindex(columns=['Locomocion','Obstruccion','Defensa','Cooperacion','Descarte'],
                               fill_value=0)
colores_acc = ['#2E86AB','#E84855','#3BB273','#F4A261','#AAAAAA']
acc_ronda.plot(kind='bar', stacked=True, ax=ax3,
               color=colores_acc, edgecolor='none', linewidth=0, width=0.7)
ax3.set_title('Distribución de acciones por ronda', fontsize=11)
ax3.set_xlabel('Ronda')
ax3.set_ylabel('Turnos')
ax3.tick_params(axis='x', rotation=0)
ax3.legend(title='Acción', fontsize=8, title_fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig('analisis_C_trueques.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊  Gráfico C guardado como analisis_C_trueques.png')

---
## 4 · Evolución del Rivalry Ratio por Ronda

**Zona roja** (Ratio < 1) = el Objetivo va ganando → ataques más probables. Los **✕ rojos** marcan ataques reales.

In [ ]:
df_evol = df[df['Rivalry_Ratio'].notna() & ~np.isinf(df['Rivalry_Ratio'])].copy()
rondas  = sorted(df_evol['Ronda_ID'].unique())
n       = len(rondas)

# 11 rondas → 4 columnas, 3 filas
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 3.8*rows), squeeze=False)
fig.suptitle(
    'Evolución del Rivalry Ratio (DY/DA) — 11 Rondas\n'
    'Zona roja = Objetivo va ganando | ✕ = Ataque | 3 jugadores',
    fontsize=13, fontweight='bold'
)

for idx, ronda in enumerate(rondas):
    r, c = divmod(idx, cols)
    ax   = axes[r][c]
    sub  = df_evol[df_evol['Ronda_ID'] == ronda]

    ax.plot(sub['Turno_Num'], sub['Rivalry_Ratio'],
            color='#2E86AB', linewidth=1.5, marker='o', markersize=3, alpha=0.8)
    ax.axhline(1, color='black', linestyle='--', linewidth=0.9, alpha=0.6)

    ax.fill_between(sub['Turno_Num'], sub['Rivalry_Ratio'], 1,
                    where=sub['Rivalry_Ratio'] < 1,
                    alpha=0.15, color='red')
    ax.fill_between(sub['Turno_Num'], 1, sub['Rivalry_Ratio'],
                    where=sub['Rivalry_Ratio'] >= 1,
                    alpha=0.08, color='green')

    atk_sub = sub[sub['Accion_Tomada'] == 'Obstruccion']
    if len(atk_sub):
        ax.scatter(atk_sub['Turno_Num'], atk_sub['Rivalry_Ratio'],
                   color='#E84855', s=60, zorder=5, marker='X', label='Ataque')

    ax.set_title(f'Ronda {ronda}  ({len(sub)} turnos c/ratio)', fontsize=10)
    ax.set_xlabel('Turno', fontsize=9)
    ax.set_ylabel('Ratio', fontsize=9)
    ax.set_ylim(bottom=0)
    ax.tick_params(labelsize=8)

# Ocultar subplots vacíos
for idx in range(n, rows*cols):
    r, c = divmod(idx, cols)
    axes[r][c].set_visible(False)

plt.tight_layout()
plt.savefig('evolucion_rivalry_ratio.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊  Gráfico de evolución guardado como evolucion_rivalry_ratio.png')

---
## 5 · Análisis Adicional — Comportamiento por Jugador

Con 11 rondas y 3 jugadores tenemos suficientes datos para analizar el **perfil competitivo individual**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Análisis de Comportamiento Individual por Jugador',
             fontsize=13, fontweight='bold', y=1.02)

# ── D-1: Acciones por jugador (barras agrupadas) ───────────────────────────
ax1 = axes[0]
acc_jug = df.groupby(['Jugador_Activo','Accion_Tomada']).size().unstack(fill_value=0)
acc_jug = acc_jug.reindex(columns=['Locomocion','Obstruccion','Defensa','Cooperacion','Descarte'],
                           fill_value=0)
# Normalizar por total de turnos del jugador
acc_jug_pct = acc_jug.div(acc_jug.sum(axis=1), axis=0) * 100
colores = ['#2E86AB','#E84855','#3BB273','#F4A261','#AAAAAA']
acc_jug_pct.plot(kind='bar', ax=ax1, color=colores,
                 edgecolor='white', linewidth=0.5, width=0.7)
ax1.set_title('Distribución de acciones (%) por jugador', fontsize=11)
ax1.set_xlabel('Jugador')
ax1.set_ylabel('% de turnos')
ax1.tick_params(axis='x', rotation=0)
ax1.legend(title='Acción', fontsize=8, title_fontsize=9)

# ── D-2: Rivalry Ratio medio por jugador (cuando atacan) ──────────────────
ax2 = axes[1]
atk_jug = df[
    (df['Accion_Tomada'] == 'Obstruccion') &
    df['Rivalry_Ratio'].notna() &
    ~np.isinf(df['Rivalry_Ratio'])
].copy()
if len(atk_jug):
    sns.boxplot(data=atk_jug, x='Jugador_Activo', y='Rivalry_Ratio',
                palette=PLAYER_COLORS, order=JUGADORES,
                linewidth=1.4, fliersize=3, ax=ax2)
    ax2.axhline(1, color='black', linestyle='--', linewidth=1.2, alpha=0.7, label='Ratio = 1')
    ax2.axhspan(0, 1, alpha=0.07, color='red')
    ax2.set_title('Rivalry Ratio al momento de atacar\n(bajo la línea = atacó al líder)', fontsize=11)
    ax2.set_xlabel('Jugador Activo')
    ax2.set_ylabel('Rivalry Ratio (DY / DA)')
    ax2.set_ylim(bottom=0)
    ax2.legend(fontsize=9)

# ── D-3: Quién recibe más ataques ─────────────────────────────────────────
ax3 = axes[2]
if len(df_atk_all):
    recibidos = df_atk_all['Jugador_Objetivo'].value_counts().reindex(JUGADORES, fill_value=0)
    lanzados  = df_atk_all['Jugador_Activo'].value_counts().reindex(JUGADORES, fill_value=0)
    x = np.arange(len(JUGADORES))
    w = 0.35
    ax3.bar(x - w/2, lanzados.values, width=w, color='#E84855', alpha=0.85,
            edgecolor='white', label='Ataques lanzados')
    ax3.bar(x + w/2, recibidos.values, width=w, color='#2E86AB', alpha=0.85,
            edgecolor='white', label='Ataques recibidos')
    for i, (l, r) in enumerate(zip(lanzados.values, recibidos.values)):
        ax3.text(i - w/2, l + 0.5, str(l), ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax3.text(i + w/2, r + 0.5, str(r), ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax3.set_xticks(x)
    ax3.set_xticklabels(JUGADORES)
    ax3.set_title('Ataques lanzados vs recibidos\npor jugador', fontsize=11)
    ax3.set_ylabel('Cantidad de ataques')
    ax3.legend(fontsize=9)

plt.tight_layout()
plt.savefig('analisis_D_jugadores.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊  Gráfico D guardado como analisis_D_jugadores.png')

---
## 6 · Resumen Final — Verificación de Hipótesis

In [ ]:
print('='*68)
print('  VERIFICACIÓN DE HIPÓTESIS — OBLEOPOLIS')
print('  Datos reales: 11 rondas · 3 jugadores · Ratio = DY / DA')
print('='*68)

# ── Hipótesis A ───────────────────────────────────────────────────────────
atk_df = df[
    (df['Accion_Tomada'] == 'Obstruccion') &
    df['Rivalry_Ratio'].notna() &
    ~np.isinf(df['Rivalry_Ratio'])
]
if len(atk_df):
    pct_a   = (atk_df['Rivalry_Ratio'] < 1).mean() * 100
    verif_a = '✅ CONFIRMADA' if pct_a >= 50 else '❌ NO CONFIRMADA'
    print(f'\nH-A  Ataques concentrados cuando Ratio < 1 (Objetivo iba ganando):')
    print(f'     {len(atk_df)} ataques analizados → {pct_a:.1f}% con Ratio < 1  →  {verif_a}')
else:
    print('\nH-A: Sin ataques con ratio calculado.')

# ── Hipótesis B ───────────────────────────────────────────────────────────
if len(df_atk_all):
    df_b = df[df['Accion_Tomada'] == 'Obstruccion'].copy()
    df_b['Es_al_lider'] = df_b.apply(
        lambda r: r['Jugador_Objetivo'] == r['Lider'] and r['Lider'] != 'Empate', axis=1
    )
    pct_b   = df_b['Es_al_lider'].mean() * 100
    verif_b = '✅ CONFIRMADA' if pct_b >= 50 else '❌ NO CONFIRMADA'
    print(f'\nH-B  Coalición implícita anti-líder:')
    print(f'     {len(df_b)} ataques → {pct_b:.1f}% van al líder  →  {verif_b}')
else:
    print('\nH-B: Sin ataques con objetivo válido.')

# ── Hipótesis C ───────────────────────────────────────────────────────────
print(f'\nH-C  Trueques aceptados en empate técnico:')
print(f'     ⚠️  Solo {len(df_truq)} trueque(s) registrado(s) — muestra insuficiente para')
print(f'     conclusiones estadísticas. Se recomienda registrar más partidas')
print(f'     con énfasis en el uso de la carta Trueque.')

print()
print('─'*68)
print('  MÉTRICAS GENERALES DE LAS PRUEBAS')
print('─'*68)
total = len(df)
for accion in ['Locomocion','Obstruccion','Defensa','Cooperacion','Descarte']:
    n_acc = (df['Accion_Tomada'] == accion).sum()
    print(f'  {accion:<15} : {n_acc:>4} turnos ({n_acc/total*100:.1f}%)')
print(f'  {"TOTAL":<15} : {total:>4} turnos en {df["Ronda_ID"].nunique()} rondas')
print()
print('  Archivos PNG generados en esta carpeta:')
print('    📊 analisis_A_rivalry_ratio.png')
print('    📊 analisis_B_alianzas.png')
print('    📊 analisis_C_trueques.png')
print('    📊 analisis_D_jugadores.png')
print('    📊 evolucion_rivalry_ratio.png')
print('='*68)